In [13]:
import pandas as pd

In [14]:
df = pd.read_csv("datasets/gpt_dataset.csv")

In [16]:
df = df[~df["Category"].isin([
    "Full Stack Developer",
    "Python Developer",
    "Mobile App Developer (iOS/Android)"
])]

In [17]:
df.head()

,Category,Resume
0,Frontend Developer,"As a seasoned Frontend Developer, I have a pro..."
1,Backend Developer,With a solid background in Backend Development...
3,Data Scientist,"With a background in Data Science, I possess a..."
4,Frontend Developer,Experienced Frontend Developer with a passion ...
5,Frontend Developer,Passionate Frontend Developer with over 4 year...


In [18]:
df["Category"].unique()

array(['Frontend Developer', 'Backend Developer', 'Data Scientist',
       'Machine Learning Engineer', 'Cloud Engineer'], dtype=object)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 263 entries, 0 to 396
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  263 non-null    object
 1   Resume    263 non-null    object
dtypes: object(2)
memory usage: 6.2+ KB


In [21]:
job_description= {
    "Frontend Developer" : """ We are looking for a talented Frontend Developer to join our development team. The ideal candidate should have strong experience in building responsive and user-friendly web applications using modern frontend technologies. 
     You will be responsible for developing interactive UI components, optimizing website performance, and collaborating closely with backend developers and designers to deliver high-quality applications. """, 

     "Backend Developer" : """We are seeking an experienced Backend Developer to design, develop, and maintain scalable server-side applications and APIs. The candidate will work closely with frontend developers, database administrators, and cloud engineers to build secure and high-performance backend systems.
     The ideal candidate should have strong proficiency in backend technologies such as Python, Node.js, Django, Flask, or Express.js. Experience with relational and non-relational databases like MySQL, PostgreSQL, or MongoDB is required.
     Knowledge of RESTful APIs, authentication systems, cloud platforms, and microservices architecture will be considered an added advantage. The candidate should be comfortable debugging complex issues and optimizing application performance.""",

    "Data Scientist" : """We are hiring a Data Scientist who is passionate about extracting insights from data and building intelligent machine learning solutions. The candidate will be responsible for data analysis, predictive modeling, feature engineering, and developing machine learning models to solve business problems.
    The ideal applicant should have strong knowledge of Python, statistics, machine learning algorithms, and data visualization techniques.
    Experience working with libraries such as Pandas, NumPy, Scikit-learn, TensorFlow, or PyTorch is highly preferred. Candidates should also be familiar with SQL, data preprocessing, and model evaluation techniques. Strong analytical thinking, communication skills, and the ability to work with large datasets are essential for this role.""",

    "Machine Learning Engineer" : """We are looking for a skilled Machine Learning Engineer to join our AI and engineering team. The candidate will be responsible for designing, developing, and deploying machine learning models to solve real-world business problems. You will work closely with data scientists, software engineers, and product teams to build scalable intelligent systems and improve model performance.
    The ideal candidate should have strong experience in Python and machine learning frameworks such as Scikit-learn, TensorFlow, or PyTorch. Hands-on experience with supervised and unsupervised learning algorithms, feature engineering, model evaluation, and data preprocessing is required. Knowledge of deep learning, NLP, computer vision, and cloud deployment tools will be considered an added advantage.
    Candidates should also have experience working with large datasets, SQL databases, APIs, and version control systems like Git. Strong problem-solving abilities, analytical thinking, and the capability to optimize machine learning pipelines for production environments are essential for this role.""",

    "Cloud Engineer" : """We are looking for a highly skilled Cloud Engineer to join our technology team and help design, implement, and maintain scalable cloud infrastructure solutions. The candidate will be responsible for managing cloud environments, automating deployment pipelines, monitoring system performance, and ensuring high availability and security of cloud-based applications.
    The ideal candidate should have strong experience with cloud platforms such as AWS, Microsoft Azure, or Google Cloud Platform (GCP). Hands-on experience with infrastructure as code tools like Terraform or CloudFormation, containerization technologies such as Docker and Kubernetes, and CI/CD pipelines is highly preferred. Candidates should also be familiar with Linux systems, networking concepts, virtualization, and cloud security best practices.
    Experience in monitoring and logging tools, server management, load balancing, and database deployment in cloud environments will be considered an added advantage. The candidate should possess strong troubleshooting skills, problem-solving abilities, and the capability to work collaboratively with development and DevOps teams in a fast-paced environment."""
}

# Positive Pairs

In [22]:
positive_pairs = []

for idx, row in df.iterrows():

    category = row["Category"]
    resume = row["Resume"]

    if category in job_description:

        jd = job_description[category]

        positive_pairs.append({
            "job_description": jd,
            "resume": resume,
            "score": 0.9,
            "type": "Positive"
        })

print(len(positive_pairs))

263


# Hard Negatives

In [24]:
hard_negative_map = {

    "Frontend Developer": [
        "Backend Developer"
    ],

    "Backend Developer": [
        "Cloud Engineer"
    ],

    "Data Scientist": [
        "Machine Learning Engineer"
    ],

    "Machine Learning Engineer": [
        "Data Scientist"
    ],

    "Cloud Engineer": [
        "Backend Developer"
    ]
}

In [25]:
hard_negatives = []

for idx, row in df.iterrows():

    jd_category = row["Category"]

    if jd_category in hard_negative_map:

        similar_categories = hard_negative_map[jd_category]

        jd = job_description[jd_category]

        for sim_cat in similar_categories:

            neg_samples = df[df["Category"] == sim_cat]

            for _, neg_row in neg_samples.iterrows():

                hard_negatives.append({

                    "job_description": jd,
                    "resume": neg_row["Resume"],
                    "score": 0.4,
                    "type": "Hard Negative"

                })

print(len(hard_negatives))

14020


# Easy Negatives

In [26]:
import random

easy_negatives = []

categories = list(df["Category"].unique())

for idx, row in df.iterrows():

    jd_category = row["Category"]

    jd = job_description[jd_category]

    unrelated = [
        c for c in categories
        if c != jd_category
        and c not in hard_negative_map.get(jd_category, [])
    ]

    random_category = random.choice(unrelated)

    neg_resume = df[df["Category"] == random_category].sample(1)

    easy_negatives.append({

        "job_description": jd,
        "resume": neg_resume.iloc[0]["Resume"],
        "score": 0.1,
        "type": "Easy Negative"

    })

print(len(easy_negatives))

263


# Combine Final ATS Dataset

In [27]:
import pandas as pd

all_pairs = (
    positive_pairs +
    hard_negatives +
    easy_negatives
)

ats_df = pd.DataFrame(all_pairs)

ats_df.head()

,job_description,resume,score,type
0,We are looking for a talented Frontend Develo...,"As a seasoned Frontend Developer, I have a pro...",0.9,Positive
1,We are seeking an experienced Backend Develope...,With a solid background in Backend Development...,0.9,Positive
2,We are hiring a Data Scientist who is passiona...,"With a background in Data Science, I possess a...",0.9,Positive
3,We are looking for a talented Frontend Develo...,Experienced Frontend Developer with a passion ...,0.9,Positive
4,We are looking for a talented Frontend Develo...,Passionate Frontend Developer with over 4 year...,0.9,Positive


In [28]:
ats_df.to_csv("ats_dataset.csv", index=False)